In [ ]:
import deepinv as dinv
import torch
import matplotlib.pyplot as plt
from deepinv.utils.demo import load_dataset
from torchvision import transforms
import numpy as np
from sapg import SAPG
from priors import WaveletPrior, RedPrior, L2, GSPnP, CombinedPrior, L1Prior
from prior_comparison import DegradedLikelihood
import tqdm
from utils import device

In [ ]:
# Set the global random seed from pytorch to ensure reproducibility of the example.
#torch.manual_seed(0)

device = torch.device('cuda') 
print(device)
# Set up the variable to fetch dataset and operators.
dataset_name = "set3c"
img_size = 64

natural_img = False
n_channels = 1

if natural_img:
    val_transform = transforms.Compose([transforms.CenterCrop(img_size), transforms.ToTensor()])

    dataset = load_dataset(dataset_name, transform=val_transform)
    im_ind = 0
    x = dataset[im_ind][0].unsqueeze(0).to(device)
    
    if n_channels == 1:
        deco = dinv.physics.Decolorize(device=device)
        x = deco(x)

else:
    sigma_img = 0.25
    x = torch.tensor(np.random.laplace(0., sigma_img, [1, 1, img_size, img_size]).astype(np.float32)).to(device)

dimx = x.shape[-1]*x.shape[-2]

filter_torch = dinv.physics.blur.gaussian_blur(sigma=(2, 2))
noise_level_img = 0.05  # Gaussian Noise standard deviation for the degradation

# The BlurFFT instance from physics enables to compute efficently backward operators with Fourier transform.
p = dinv.physics.Blur(
    img_size=(1, img_size, img_size),
    filter=filter_torch,
    device=device, padding="circular",
    noise_model=dinv.physics.GaussianNoise(sigma=noise_level_img))

# generate blurred, noisy measurement
y = p(x) 

# plot the signal, the measurement and the noise
dinv.utils.plot([x, y, y - x], ["signal", "measurement", "diff"], rescale_mode='clip')

In [ ]:
# Lipschitz constant of nabla f
L_f = p.compute_norm(x0=torch.randn_like(x), tol=1e-5) / noise_level_img**2 / 2.  # divide by 2 because std is sqrt(2)sigma

# regularization parameter of proximity operator 
lam_reg = min(1/L_f, 2.)   
# Lipschitz constant of grad g
L_g = 1 / lam_reg 
L =  L_f + L_g

# Stepsize of MCMC algorithm
gamma = 0.98*1/L
print("gamma: {},  lam: {}".format(gamma, lam_reg))


### Define the priors and associated constants

In [ ]:
nval = 7 # odd
valtests = torch.linspace(2., 6., nval) 
hist = np.zeros(nval)
nb_steps, burnin_ratio = 10, 0.6
mod_comps = []
noise = torch.randn_like(y)*noise_level_img
for i in range(nval):
    print("alpha={}".format(valtests[i]))
    g = L1Prior(valtests[i])
    
    mod_comps.append(DegradedLikelihood(y, g, p, noise_level_img, gamma, X_init=p.A_A_adjoint(y).to(device).clone(),
                                        lam_reg=lam_reg, project=None, noise=noise))
    # X_post_trace1, post_hist1, X_post_trace2, post_hist2 =
    lik1  = mod_comps[i].compute_test(nb_steps, burnin_ratio=burnin_ratio, log_stats=False, thinning=50)
    print("likelihood: ", lik1)
    hist[i] = lik1


In [ ]:
print(hist)
ind_exact = np.where(valtests == 4.)[0][0]
mlogpost = np.array([hist[i].item() for i in range(len(hist))])
plt.plot(valtests, mlogpost, '.-')
plt.xlabel(r'$\theta$'), plt.ylabel('r')
#plt.yscale('log')
print(valtests)
plt.vlines(1./sigma_img, np.min(mlogpost), np.max(mlogpost),color='red')
plt.savefig('figs/test1c.pdf')

In [ ]:
hist = np.zeros(nval)
nb_steps = 10
mod_comps = []
for i in range(nval):
    print("alpha={}".format(valtests[i]))
    g = L1Prior(torch.tensor(valtests[i]))
    
    mod_comps.append(DegradedLikelihood(y, g, p, noise_level_img, gamma, X_init=p.A_A_adjoint(y).to(device).clone(),
                                        lam_reg=lam_reg, project=None))
    # X_post_trace1, post_hist1, X_post_trace2, post_hist2 =
    lik1  = mod_comps[i].compute_test2(nb_steps, thinning=50, burnin_ratio=0.5)
    print("likelihood: ", lik1)
    hist[i] = lik1


In [ ]:
print(hist)
ind_exact = np.where(valtests == 4.)[0][0]
plt.plot(valtests, hist, '.-')
plt.xlabel(r'$\theta$'), plt.ylabel('r')
plt.vlines(1./sigma_img, np.min(hist), np.max(hist),color='red')
plt.savefig('figs/test2.pdf')